Importing all needed libaries like panda and sklearn

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
)

RANDOM_STATE = 20   # same seed as the clustering notebook
os.makedirs("processed", exist_ok=True)

Load Dataset

In [2]:
data = pd.read_csv("processed/lga_yearly_aggregates_clustering.csv")
print(data.shape)
print("LGAs:", data["LGA"].nunique(), "| Years:", sorted(data["Year"].unique()))
print("Missing values:", int(data.isna().sum().sum()))
data.head()

(645, 10)
LGAs: 129 | Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Missing values: 0


,LGA,Year,Population,severe_count,non_severe_count,Wet-weather %,Rural-road %,Crash rate per 100k,Severe %,total_crashes
0,Albury,2020,55466,73,50,12.195122,13.821138,221.757473,59.349593,123
1,Armidale,2020,29600,47,10,24.561404,56.140351,192.567568,82.456140,57
2,Ballina,2020,45663,58,19,22.077922,23.376623,168.626678,75.324675,77
3,Balranald,2020,2257,5,1,0.000000,83.333333,265.839610,83.333333,6
4,Bathurst,2020,43444,81,28,23.853211,32.110092,250.897707,74.311927,109


build labels & features

In [3]:
RATE_COL = "Crash rate per 100k"
FEATURES = ["Population", "Severe %", "Wet-weather %", "Rural-road %"]

train_df = data[data["Year"] <= 2023].copy()
test_df = data[data["Year"] == 2024].copy()

# Threshold from the TRAINING years only
risk_threshold = train_df[RATE_COL].median()

train_df["High_risk"] = (train_df[RATE_COL] >= risk_threshold).astype(int)
test_df["High_risk"] = (test_df[RATE_COL] >= risk_threshold).astype(int)

# NumPy arrays with float dtype
X_train = train_df[FEATURES].astype(float).to_numpy()
X_test  = test_df[FEATURES].astype(float).to_numpy()
y_train = train_df["High_risk"].to_numpy()
y_test  = test_df["High_risk"].to_numpy()

# LGA names are only used to keep each LGA inside one cross-validation fold
groups_train = train_df["LGA"].to_numpy

print(f"High-risk threshold (median training crash rate): {risk_threshold:.1f} per 100k")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Share of High risk: train {y_train.mean():.1%} | test {y_test.mean():.1%}")

High-risk threshold (median training crash rate): 254.9 per 100k
Train: (516, 4) | Test: (129, 4)
Share of High risk: train 50.0% | test 46.5%


 Logistic Regression & Model Evaluation

In [4]:
# Initialize and train the Logistic Regression model
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
)
log_reg.fit(X_train, y_train)

# Predict on the test data
y_pred_log_reg = log_reg.predict(X_test)
y_prob_log_reg = log_reg.predict_proba(X_test)[:, 1]

# Evaluate Logistic Regression
print("Logistic Regression Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log_reg):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_log_reg):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_log_reg):.2f}")
print(f"F1-Score: {f1_score(y_test, y_pred_log_reg):.2f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log_reg))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log_reg))

groups_train = train_df["LGA"].to_numpy()


Logistic Regression Evaluation:
Accuracy: 0.74
Precision: 0.70
Recall: 0.78
F1-Score: 0.74

Confusion Matrix:
[[49 20]
 [13 47]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.71      0.75        69
           1       0.70      0.78      0.74        60

    accuracy                           0.74       129
   macro avg       0.75      0.75      0.74       129
weighted avg       0.75      0.74      0.74       129



In [5]:
# Naive baseline on the same test set
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy accuracy: {dummy.score(X_test, y_test):.4f}")

# ROC-AUC on the test set
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_prob_log_reg):.4f}")

# Grouped CV so an LGA's rows never span folds
cv = GroupKFold(n_splits=5)
cv_scores = cross_val_score(
    log_reg, X_train, y_train,
    groups=groups_train, cv=cv,
    scoring="roc_auc",
)
print(f"CV ROC-AUC:     {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

Dummy accuracy: 0.5349
Test ROC-AUC:   0.8452
CV ROC-AUC:     0.821 ± 0.011


Save Logistic Regression for visulization

In [6]:
# Save Logistic Regression for visulization
try:
    perm
except NameError:
    perm = permutation_importance(
        log_reg, X_test, y_test,
        n_repeats=20, random_state=RANDOM_STATE, scoring="roc_auc",
    )

# 1. Per-LGA test predictions
test_out = test_df[["LGA", "Year"]].copy()
test_out["y_true"] = y_test
test_out["y_pred"] = y_pred_log_reg
test_out["y_prob"] = y_prob_log_reg
test_out.to_csv("processed/logreg_test_predictions.csv", index=False)

# 2. Headline metrics (test set + dummy baseline)
metrics = pd.DataFrame([{
    "accuracy":       accuracy_score(y_test, y_pred_log_reg),
    "precision":      precision_score(y_test, y_pred_log_reg),
    "recall":         recall_score(y_test, y_pred_log_reg),
    "f1":             f1_score(y_test, y_pred_log_reg),
    "roc_auc":        roc_auc_score(y_test, y_prob_log_reg),
    "dummy_accuracy": dummy.score(X_test, y_test),
    "cv_roc_auc_mean": cv_scores.mean(),
    "cv_roc_auc_std":  cv_scores.std(),
}])
metrics.to_csv("processed/logreg_metrics.csv", index=False)

# 3. CV fold scores
pd.DataFrame({
    "fold": range(1, len(cv_scores) + 1),
    "cv_roc_auc": cv_scores,
}).to_csv("processed/logreg_cv_scores.csv", index=False)

# 4. Coefficients
pd.DataFrame({
    "feature": FEATURES,
    "coefficient": log_reg.named_steps["logisticregression"].coef_[0],
}).to_csv("processed/logreg_coefficients.csv", index=False)

# 5. Permutation importance
pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std,
}).to_csv("processed/logreg_permutation_importance.csv", index=False)

print("Saved logistic-regression outputs to processed")

Saved logistic-regression outputs to processed
